In [1]:
# !pip install codecarbon
# !pip install -e .

In [1]:
import pandas as pd
import pickle
from trust_library.core import TrustEvaluator
from trust_library.factsheet import (
    load_factsheet_default,
    load_factsheet,
    save_factsheet, 
    create_factsheet_interactive,
    create_factsheet,
)

import matplotlib.pyplot as plt
import seaborn as sns
import json

from trust_library.utils import to_json_safe
import plotly.express as px

if __name__ == "__main__":

    # factsheet = create_factsheet_interactive()
    # print(load_factsheet_default()) 
    # fs = create_factsheet({
    # "general": {
    #     "target_column": "Target"
    # },
    # "fairness": {
    #     "protected_feature": "Group",
    #     "protected_values": [1],      # Valor que identifica al grupo desprotegido
    #     "favorable_outcomes": [1]     # Valor que identifica el éxito
    # }
    # })
    # save_factsheet(fs, "factsheet.json")

    factsheet = load_factsheet_default() #load_factsheet("factsheet.json")
    
    with open("model.pkl", "rb") as f:
        loaded_model = pickle.load(f)

    train_loaded = pd.read_csv("train.csv")
    test_loaded = pd.read_csv("test.csv")

    evaluator = TrustEvaluator(
        model=loaded_model,
        train_data=train_loaded,
        test_data=test_loaded,
        factsheet=factsheet
    )
    results = evaluator.evaluate()

    print("\n=== RESULTADOS FINAL ===")
    print(f"Global Trust Score: {results['trust_score']}")
    print(f"Pillar Scores:\n{json.dumps(results['pillar_score'], indent=4)}")
    print("\nDetalles (Scores brutos):\n" + json.dumps(results['details'], indent=4))

    print("\nPropiedades calculadas:\n" + json.dumps(to_json_safe(results['properties']), indent=4))
    print("\n=== EXPLICACIÓN DEL SCORE ===")
    print(json.dumps(results['explanation'], indent=4))

Computing Fairness metrics...
Computing Accountability metrics...
Computing Privacy metrics...


[codecarbon WARNING @ 22:23:58] Multiple instances of codecarbon are allowed to run at the same time.


Computing Sustainability metrics...
Computing Explainability metrics...
Computing Robustness metrics...

=== RESULTADOS FINAL ===
Global Trust Score: 2.3
Pillar Scores:
{
    "fairness": 2.9,
    "accountability": 1.7,
    "privacy": 5.0,
    "sustainability": 1.0,
    "explainability": 3.2,
    "robustness": 0
}

Detalles (Scores brutos):
{
    "fairness": {
        "underfitting": NaN,
        "overfitting": NaN,
        "class_balance": NaN,
        "statistical_parity_difference": 1,
        "disparate_impact": 5,
        "equal_opportunity_difference": 3,
        "average_odds_difference": 4,
        "accuracy_parity": 5,
        "predictive_parity": NaN,
        "treatment_equality": NaN,
        "calibration_gap": NaN,
        "well_calibration_error": NaN,
        "generalized_entropy_index": NaN,
        "theil_index": 4,
        "coefficient_of_variation": 1,
        "individual_consistency": 1,
        "class_imbalance": NaN,
        "kl_divergence": NaN,
        "smoothed_e

In [3]:
subset = evaluator.evaluate_pillars(["fairness", "privacy"])
# Imprimimos subset formateado
print("\n=== SUBSET DE PILLARS (Fairness y Privacy) ===")
print(json.dumps(subset, indent=4))

Computing Fairness metrics...
Computing Privacy metrics...

=== SUBSET DE PILLARS (Fairness y Privacy) ===
{
    "fairness": {
        "score": 2.8,
        "metrics": {
            "underfitting": NaN,
            "overfitting": NaN,
            "class_balance": NaN,
            "statistical_parity_difference": 1,
            "disparate_impact": 5,
            "equal_opportunity_difference": 3,
            "average_odds_difference": 4,
            "accuracy_parity": 5,
            "predictive_parity": NaN,
            "treatment_equality": NaN,
            "calibration_gap": NaN,
            "well_calibration_error": NaN,
            "generalized_entropy_index": NaN,
            "theil_index": NaN,
            "coefficient_of_variation": NaN,
            "individual_consistency": 1,
            "class_imbalance": NaN,
            "kl_divergence": NaN,
            "smoothed_edf": NaN,
            "bias_amplification": NaN,
            "cohens_d": 1
        },
        "properties": {
  

In [4]:
    # with open("trust_evaluation_result.json", "r") as f:
    #     trust_results = json.load(f)
    # pillar_scores = trust_results["pillar_score"]

    # df_pillars = pd.DataFrame(
    #     list(pillar_scores.items()),
    #     columns=["Pillar", "Score"]
    # )

    # plt.figure(figsize=(8, 5))
    # sns.barplot(data=df_pillars, x="Pillar", y="Score")
    # plt.title("Trust Pillar Scores")
    # plt.ylim(0, 5)
    # plt.tight_layout()
    # plt.show()

    # details = trust_results["details"]

    # for pillar, metrics in details.items():
    #     df = pd.DataFrame(
    #         list(metrics.items()),
    #         columns=["Metric", "Score"]
    #     )

    #     plt.figure(figsize=(10, 6))
    #     sns.barplot(data=df, x="Score", y="Metric")
    #     plt.title(f"{pillar.capitalize()} Metrics")
    #     plt.xlim(0, 5)
    #     plt.tight_layout()
    #     plt.show()

    # === Cargar datos ===
    # with open("trust_evaluation_result.json", "r") as f:
    #     trust_results = json.load(f)

In [5]:
# === Cargar datos ===

pillar_scores = results["pillar_score"]
details = results["details"]

# === Gráfico de barras de pilares ===
df_pillars = pd.DataFrame(list(pillar_scores.items()), columns=["Pillar", "Score"])
fig_pillars = px.bar(
    df_pillars,
    x="Pillar",
    y="Score",
    text="Score",
    color="Score",
    color_continuous_scale="Viridis",
    range_y=[0, 5],
    title="Trust Pillar Scores"
)
fig_pillars.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig_pillars.show()  # Esto abre un navegador con el gráfico interactivo

# === Gráficos por métricas individuales de cada pilar ===
for pillar, metrics in details.items():
    df_metrics = pd.DataFrame(list(metrics.items()), columns=["Metric", "Score"])
    df_metrics = df_metrics.sort_values("Score")  # Para que las barras horizontales estén ordenadas
    fig = px.bar(
        df_metrics,
        x="Score",
        y="Metric",
        orientation="h",
        text="Score",
        color="Score",
        color_continuous_scale="Viridis",
        range_x=[0, 5],
        title=f"{pillar.capitalize()} Metrics"
    )
    fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
    fig.show()  # Abre cada gráfico en una ventana o pestaña del navegador

In [6]:
import plotly.graph_objects as go

# === Cargar datos ===
pillar_scores = results["pillar_score"]

# Convertir a listas
categories = list(pillar_scores.keys())
values = list(pillar_scores.values())

# Cerrar el polígono (importante en radar)
categories += [categories[0]]
values += [values[0]]

# Crear figura
fig_radar = go.Figure()

fig_radar.add_trace(go.Scatterpolar(
    r=values,
    theta=categories,
    fill='toself',
    name='Model Score',
    
))

fig_radar.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 5]
        )
    ),
    title="Trust Pillar Radar Chart",
    showlegend=False
)

fig_radar.show()